# Cold-Start Evaluation
**Hybrid Music Recommendation System — DSCI 441, Spring 2026**

Evaluates four models in the cold-start scenario — each user is known only
by a single song (their most-played), with all other interactions held out.

Models:
- `PopularityBaseline` — ignores seed, returns globally popular songs
- `CFColdStart` — CF has no user vector; degenerates to popularity (benchmarked honestly)
- `Content-only` — k-NN of seed song
- `Hybrid` — cold-start path: `user_id=None`, `seed_song=seed`

Uses the **same 1000 users** as `final_evaluation.ipynb` (loaded from
`results/test_users.pkl`) so warm vs cold comparisons are valid.

**Methodology note:**  
Held-out items are drawn from each user's full MSD listening history, which
extends well beyond the 7,611-song content catalog. This penalises Content and
Hybrid in absolute terms, but the same penalty applies to all models, so
cross-model comparisons remain fair. The key question is relative ranking, not
absolute magnitude.

---
## Section 1 — Setup

In [ ]:
import os, sys
os.environ['OPENBLAS_NUM_THREADS'] = '1'

from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import joblib

from src.data import (
    load_msd_triplets, load_msd_metadata, load_spotify_tracks,
    match_datasets, build_metadata_catalog,
)
from src.models import CFModel, ContentModel, PopularityBaseline, CFColdStart
from src.hybrid import HybridRecommender
from src.evaluation import (
    evaluate_model, bootstrap_ci, paired_bootstrap_test,
    cold_start_test_split,
)

MODELS_DIR  = Path('../models')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

K_VALUES = [5, 10, 20]

In [ ]:
# Load CF model
cf = CFModel.load(MODELS_DIR)
cf_song_to_idx = {sid: idx for idx, sid in cf._idx_to_song.items()}
print(f'CF: {len(cf._user_to_idx):,} users, {len(cf._idx_to_song):,} songs')

In [ ]:
# Load metadata
msd_meta = load_msd_metadata()
spotify  = load_spotify_tracks()
matched  = match_datasets(spotify, msd_meta)
meta_cat = build_metadata_catalog(matched)
print(f'metadata_catalog: {len(meta_cat):,} song_ids')

In [ ]:
# Load Content model
cm = ContentModel.load(MODELS_DIR)
print(f'Content k-NN index: {len(cm._songid_to_idx):,} songs')

In [ ]:
# Fit PopularityBaseline on same filtered triplets
_TRIP_CACHE = RESULTS_DIR / 'triplets_cache.pkl'
if _TRIP_CACHE.exists():
    triplets = joblib.load(_TRIP_CACHE)
    print('Loaded triplets from cache')
else:
    triplets = load_msd_triplets()
    joblib.dump(triplets, _TRIP_CACHE)
    print('Loaded and cached triplets')

pop    = PopularityBaseline().fit(triplets)
cf_cs  = CFColdStart(cf, pop)
hybrid = HybridRecommender(cf, cm, meta_cat, popularity_model=pop, alpha_strategy='adaptive')
print('All models ready')

In [ ]:
# Load the EXACT same 1000 users from final_evaluation.ipynb
# (saved to results/test_users.pkl in that notebook's Section 2)
sampled_users = joblib.load(RESULTS_DIR / 'test_users.pkl')
print(f'Loaded {len(sampled_users)} canonical test users from test_users.pkl')

---
## Section 2 — Cold-Start Test Split

**Design choice:** each user is given exactly one known song as the "seed"
for cold-start inference. The seed is the user's most-played song *from the
Content model's k-NN index* — not necessarily their overall most-played song.

We restrict cold-start eval to users with at least one content-catalog song, so
all four models (Popularity, CFColdStart, Content, Hybrid) can be meaningfully
compared on the same protocol. Users with zero content-catalog interactions are
excluded; their count is reported below.

The held-out set is all other songs the user listened to, excluding the seed.
Held-out items are NOT restricted to the content catalog — this penalty applies
fairly to all models, keeping cross-model comparisons valid.

In [ ]:
_CS_SPLIT_CACHE = RESULTS_DIR / 'cold_start_split.pkl'

if _CS_SPLIT_CACHE.exists():
    cs_data = joblib.load(_CS_SPLIT_CACHE)
    print(f'Loaded cold-start split from cache ({len(cs_data)} users)')
    print(f'  ({len(sampled_users) - len(cs_data)} of the 1000 canonical users excluded '
          f'— no content-catalog seed available)')
else:
    content_ids = set(cm._songid_to_idx.keys())
    sampled_set = set(sampled_users)

    plays = (
        triplets[triplets['uid'].isin(sampled_set)]
        .groupby(['uid', 'sid'])['count']
        .sum()
    )
    users_with_plays = set(plays.index.get_level_values(0))

    cs_data = {}
    excluded_no_content = 0
    for uid in sampled_users:
        if uid not in users_with_plays:
            excluded_no_content += 1
            continue
        # String-index so isin() works cleanly against content_ids (str keys)
        user_plays = plays.loc[uid].sort_values(ascending=False)
        user_plays.index = user_plays.index.astype(str)
        # Restrict seed candidates to content catalog
        content_plays = user_plays[user_plays.index.isin(content_ids)]
        if len(content_plays) == 0:
            excluded_no_content += 1
            continue
        seed_song = content_plays.index[0]          # most-played in content catalog
        held_out  = {s for s in user_plays.index if s != seed_song}
        cs_data[uid] = {
            'seed_song_id':   seed_song,
            'held_out_songs': held_out,
            'history_hidden': True,
        }

    joblib.dump(cs_data, _CS_SPLIT_CACHE)
    print(f'Built and cached cold-start split ({len(cs_data)} users)')
    print(f'Excluded {excluded_no_content} users with zero content-catalog interactions')

# Diagnostics
content_ids   = set(cm._songid_to_idx.keys())
seeds_in_ct   = sum(1 for d in cs_data.values() if d['seed_song_id'] in content_ids)
heldout_in_ct = sum(
    1 for d in cs_data.values()
    if any(s in content_ids for s in d['held_out_songs'])
)
held_sizes = [len(d['held_out_songs']) for d in cs_data.values()]

print(f'\nCold-start split stats:')
print(f'  Total users:                           {len(cs_data)}')
print(f'  Seeds in content catalog:              {seeds_in_ct} '
      f'({100*seeds_in_ct/len(cs_data):.1f}%)  <- should be 100%')
print(f'  Users with any held-out in content:    {heldout_in_ct} '
      f'({100*heldout_in_ct/len(cs_data):.1f}%)  <- upper bound on content hits')
print(f'  Held-out songs per user:  '
      f'median={np.median(held_sizes):.0f}, '
      f'min={min(held_sizes)}, max={max(held_sizes)}')


In [ ]:
# cs_users is a subset of warm_users (filtered to users with content-catalog seeds)
warm_users = set(sampled_users)
cs_users   = set(cs_data.keys())
extra      = cs_users - warm_users
assert not extra, f'{len(extra)} cs_users not in warm_users — bug!'
excluded = warm_users - cs_users
print(f'Cold-start eval: {len(cs_users)} users  '
      f'(excluded {len(excluded)} of 1000 with no content-catalog seed)')


In [ ]:
# Sanity check: genre-conditioned RRF cold-start, 3 users
cs_keys = list(cs_data.keys())
test_users = [cs_keys[0], cs_keys[100], cs_keys[500]]

n_global_fallback = 0
any_both_gte2 = False

for uid in test_users:
    seed = cs_data[uid]['seed_song_id']
    hy_out = hybrid.recommend(user_id=None, seed_song=seed, k=10)

    fallback = hy_out['fallback_used'].iloc[0]
    if fallback == 'global':
        n_global_fallback += 1

    src_counts = hy_out['source'].value_counts().to_dict()
    n_both = src_counts.get('both', 0)
    if n_both >= 2:
        any_both_gte2 = True

    # Seed genre for reporting
    genre_row = meta_cat.loc[meta_cat['song_id'] == seed, 'track_genre']
    genre = genre_row.iloc[0] if not genre_row.empty else 'Unknown'

    print(f'User idx={cs_keys.index(uid):>3}  seed={seed}')
    print(f'  genre={genre!r}  fallback={fallback}')
    print(f'  source breakdown: {src_counts}')
    print(hy_out[['song_id', 'title', 'rrf_score',
                  'content_rank', 'popularity_rank', 'source']].to_string(index=False))
    print()

print(f'Global fallbacks: {n_global_fallback}/3')
print(f'Any user with >= 2 "both" songs: {any_both_gte2}')


---
## Section 3 — Cold-Start Evaluation

Define one `recommend_fn(user_id, k) -> list[str]` per model.
Each wrapper reads the user's seed from `cs_data` and calls the model in cold-start mode.
Exceptions are caught inside `evaluate_model` and treated as empty recommendations.

All four functions share the same signature so `evaluate_model` can call them uniformly.
Results are cached to avoid re-running the ~3500-user × 4-model eval on re-runs.

In [ ]:
_CS_PER_USER = RESULTS_DIR / 'cold_start_per_user.csv'

if _CS_PER_USER.exists():
    per_user_cs = pd.read_csv(_CS_PER_USER)
    print(f'Loaded cold-start per-user metrics from cache ({len(per_user_cs):,} rows)')
else:
    test_data_cs = {uid: d['held_out_songs'] for uid, d in cs_data.items()}

    pop_fn    = lambda uid, k: pop.recommend(k=k)['song_id'].tolist()
    cf_cs_fn  = lambda uid, k: cf_cs.recommend(k=k)['song_id'].tolist()
    content_fn = lambda uid, k: (
        cm.recommend(cs_data[uid]['seed_song_id'], k=k)['song_id'].tolist()
    )
    hybrid_fn  = lambda uid, k: (
        hybrid.recommend(
            user_id=None,
            seed_song=cs_data[uid]['seed_song_id'],
            k=k,
        )['song_id'].tolist()
    )

    model_fns = {
        'Popularity':   pop_fn,
        'CFColdStart':  cf_cs_fn,
        'Content':      content_fn,
        'Hybrid':       hybrid_fn,
    }

    frames = []
    for name, fn in model_fns.items():
        print(f'Evaluating {name}...', end=' ', flush=True)
        df = evaluate_model(fn, test_data_cs, k_values=K_VALUES)
        df['model'] = name
        frames.append(df)
        print('done')

    per_user_cs = pd.concat(frames, ignore_index=True)
    per_user_cs.to_csv(_CS_PER_USER, index=False)
    print(f'\nSaved cold-start per-user metrics ({len(per_user_cs):,} rows)')


In [ ]:
# Fraction of users where Hybrid top-10 != Content top-10
n_differ = 0
sample_uids = list(cs_data.keys())[:200]  # spot-check 200 users
for uid in sample_uids:
    seed = cs_data[uid]['seed_song_id']
    ct_ids = set(cm.recommend(seed, k=10)['song_id'].tolist())
    hy_ids = set(hybrid.recommend(user_id=None, seed_song=seed, k=10)['song_id'].tolist())
    if ct_ids != hy_ids:
        n_differ += 1
print(f'Hybrid differs from Content: {n_differ}/{len(sample_uids)} users '
      f'({100*n_differ/len(sample_uids):.1f}%)')


---
## Section 4 — Aggregate Metrics with Bootstrap CIs

In [ ]:
_CS_METRICS = RESULTS_DIR / 'cold_start_metrics.csv'

model_order = ['Popularity', 'CFColdStart', 'Content', 'Hybrid']
metric_order = ['ndcg', 'hit_rate', 'recall', 'precision']

rows = []
for model in model_order:
    sub = per_user_cs[per_user_cs['model'] == model]
    for k in K_VALUES:
        for metric in metric_order:
            scores = sub[(sub['k'] == k) & (sub['metric'] == metric)]['value'].values
            point, lo, hi = bootstrap_ci(scores)
            rows.append({
                'model': model, 'k': k, 'metric': metric,
                'mean': point, 'ci_low': lo, 'ci_high': hi,
            })

cs_metrics = pd.DataFrame(rows)
cs_metrics.to_csv(_CS_METRICS, index=False)
print('Saved cold_start_metrics.csv')

# Print markdown table for k=10
print('\n### Cold-Start Metrics @ k=10 (bootstrap 95% CI, n=871 users)\n')
print(f'{"Model":<14} {"NDCG@10":<22} {"HitRate@10":<22} {"Recall@10":<22} {"Precision@10":<22}')
print('-' * 104)
for model in model_order:
    parts = [f'{model:<14}']
    for metric in ['ndcg', 'hit_rate', 'recall', 'precision']:
        r = cs_metrics[(cs_metrics['model'] == model) &
                       (cs_metrics['k'] == 10) &
                       (cs_metrics['metric'] == metric)].iloc[0]
        parts.append(f'{r["mean"]:.4f} [{r["ci_low"]:.4f}, {r["ci_high"]:.4f}]')
    print('  '.join(f'{p:<22}' for p in parts))

# Flag any model whose CI includes 0
print('\n--- CI overlapping zero check ---')
for _, row in cs_metrics[(cs_metrics['k'] == 10)].iterrows():
    if row['ci_low'] <= 0.0:
        print(f'  WARNING: {row["model"]} {row["metric"]}@10  '
              f'CI=[{row["ci_low"]:.4f}, {row["ci_high"]:.4f}]  includes zero')


---
## Section 4b — Long-Tail Evaluation

**Motivation:** Standard hit-rate metrics on heavy-listener test sets are dominated by
popularity bias — the globally most-played songs appear in nearly every user's listening
history regardless of recommender quality (Steck 2011, *Item Popularity and Recommendation
Accuracy*). This gives Popularity an unfair advantage that has nothing to do with
recommendation relevance.

**Protocol:** Exclude the top-100 globally most-played songs from each user's held-out
ground truth before scoring. Users whose remaining held-out set is empty are dropped.
All four models are re-scored on the filtered ground truth.
Recommendation lists are **not** filtered — the models still recommend globally popular
songs, but they no longer receive credit for hitting mega-hits in the ground truth.

In [ ]:
_LT_METRICS = RESULTS_DIR / 'cold_start_metrics_longtail.csv'
_LT_PER_USER = RESULTS_DIR / 'cold_start_per_user_longtail.csv'

# Compute global top-100 songs by total play count
top100_global = set(
    triplets.groupby('sid')['count'].sum()
    .sort_values(ascending=False)
    .head(100)
    .index.astype(str)
)
print(f'Top-100 global songs computed. Example: {list(top100_global)[:3]}')

# Build long-tail test data: exclude top-100 from held-out ground truth
test_data_lt = {}
for uid, d in cs_data.items():
    lt_held_out = d['held_out_songs'] - top100_global
    if lt_held_out:  # skip users with nothing left
        test_data_lt[uid] = lt_held_out

n_dropped = len(cs_data) - len(test_data_lt)
print(f'Long-tail test users: {len(test_data_lt)} retained, {n_dropped} dropped (empty held-out after filter)')
print(f'Median held-out size (LT): {int(np.median([len(v) for v in test_data_lt.values()]))}')


In [ ]:
if _LT_PER_USER.exists():
    per_user_lt = pd.read_csv(_LT_PER_USER)
    print(f'Loaded long-tail per-user metrics from cache ({len(per_user_lt):,} rows)')
else:
    # Reuse same recommend_fns from Section 3 (same models, same cold-start logic)
    # Only the ground truth (test_data_lt) changes.
    pop_fn    = lambda uid, k: pop.recommend(k=k)['song_id'].tolist()
    cf_cs_fn  = lambda uid, k: cf_cs.recommend(k=k)['song_id'].tolist()
    content_fn = lambda uid, k: (
        cm.recommend(cs_data[uid]['seed_song_id'], k=k)['song_id'].tolist()
    )
    hybrid_fn  = lambda uid, k: (
        hybrid.recommend(user_id=None, seed_song=cs_data[uid]['seed_song_id'], k=k)['song_id'].tolist()
    )

    model_fns = {
        'Popularity':  pop_fn,
        'CFColdStart': cf_cs_fn,
        'Content':     content_fn,
        'Hybrid':      hybrid_fn,
    }

    frames = []
    for name, fn in model_fns.items():
        print(f'Evaluating {name} (LT)...', end=' ', flush=True)
        df = evaluate_model(fn, test_data_lt, k_values=K_VALUES)
        df['model'] = name
        frames.append(df)
        print('done')

    per_user_lt = pd.concat(frames, ignore_index=True)
    per_user_lt.to_csv(_LT_PER_USER, index=False)
    print(f'\nSaved long-tail per-user metrics ({len(per_user_lt):,} rows)')


In [ ]:
model_order  = ['Popularity', 'CFColdStart', 'Content', 'Hybrid']
metric_order = ['ndcg', 'hit_rate', 'recall', 'precision']

rows = []
for model in model_order:
    sub = per_user_lt[per_user_lt['model'] == model]
    for k in K_VALUES:
        for metric in metric_order:
            scores = sub[(sub['k'] == k) & (sub['metric'] == metric)]['value'].values
            point, lo, hi = bootstrap_ci(scores)
            rows.append({'model': model, 'k': k, 'metric': metric,
                         'mean': point, 'ci_low': lo, 'ci_high': hi})

lt_metrics = pd.DataFrame(rows)
lt_metrics.to_csv(_LT_METRICS, index=False)
print('Saved cold_start_metrics_longtail.csv\n')

# Print table at k=10
n_lt = len(test_data_lt)
print(f'### Long-Tail Cold-Start Metrics @ k=10 (top-100 excluded, n={n_lt} users)\n')
print(f'{"Model":<14} {"NDCG@10_LT":<24} {"HitRate@10_LT":<24} {"Recall@10_LT":<24} {"Prec@10_LT":<24}')
print('-' * 110)
for model in model_order:
    parts = [f'{model:<14}']
    for metric in ['ndcg', 'hit_rate', 'recall', 'precision']:
        r = lt_metrics[(lt_metrics['model'] == model) &
                       (lt_metrics['k'] == 10) &
                       (lt_metrics['metric'] == metric)].iloc[0]
        parts.append(f'{r["mean"]:.4f} [{r["ci_low"]:.4f},{r["ci_high"]:.4f}]')
    print('  '.join(f'{p:<24}' for p in parts))

# Model ranking comparison
print('\n--- Model ranking (NDCG@10): standard vs long-tail ---')
std_rank = (
    cs_metrics[(cs_metrics['k'] == 10) & (cs_metrics['metric'] == 'ndcg')]
    .set_index('model')['mean']
    .reindex(model_order)
)
lt_rank = (
    lt_metrics[(lt_metrics['k'] == 10) & (lt_metrics['metric'] == 'ndcg')]
    .set_index('model')['mean']
    .reindex(model_order)
)
for model in model_order:
    print(f'  {model:<14}  standard={std_rank[model]:.4f}   long-tail={lt_rank[model]:.4f}')

# CI overlap check: Hybrid vs Popularity under long-tail
print('\n--- Hybrid vs Popularity CI overlap (NDCG@10 LT) ---')
hy = lt_metrics[(lt_metrics['model']=='Hybrid') & (lt_metrics['k']==10) & (lt_metrics['metric']=='ndcg')].iloc[0]
pp = lt_metrics[(lt_metrics['model']=='Popularity') & (lt_metrics['k']==10) & (lt_metrics['metric']=='ndcg')].iloc[0]
print(f'  Hybrid:     {hy["mean"]:.4f} [{hy["ci_low"]:.4f}, {hy["ci_high"]:.4f}]')
print(f'  Popularity: {pp["mean"]:.4f} [{pp["ci_low"]:.4f}, {pp["ci_high"]:.4f}]')
if hy['ci_high'] < pp['ci_low']:
    print('  => CIs do not overlap. Popularity > Hybrid even under long-tail.')
elif hy['ci_low'] > pp['ci_high']:
    print('  => CIs do not overlap. Hybrid > Popularity under long-tail.')
else:
    print('  => CIs overlap. Difference not significant under long-tail.')


---
## Section 5 — Hypothesis Tests

Paired bootstrap test (10,000 resamples) on per-user NDCG@10.  
Run on both standard and long-tail cold-start protocols.
H0: E[model_A - model_B] = 0 (two-sided).

In [ ]:
_HT_CSV = RESULTS_DIR / 'cold_start_hypothesis_tests.csv'

def get_ndcg10(per_user_df, model_name):
    """Extract per-user NDCG@10, sorted by user_id for alignment."""
    return (
        per_user_df[
            (per_user_df['model'] == model_name) &
            (per_user_df['k'] == 10) &
            (per_user_df['metric'] == 'ndcg')
        ]
        .sort_values('user_id')['value']
        .values
    )

# Standard cold-start
std_pop  = get_ndcg10(per_user_cs, 'Popularity')
std_cfcs = get_ndcg10(per_user_cs, 'CFColdStart')
std_ct   = get_ndcg10(per_user_cs, 'Content')
std_hy   = get_ndcg10(per_user_cs, 'Hybrid')

# Long-tail cold-start
lt_pop   = get_ndcg10(per_user_lt, 'Popularity')
lt_cfcs  = get_ndcg10(per_user_lt, 'CFColdStart')
lt_ct    = get_ndcg10(per_user_lt, 'Content')
lt_hy    = get_ndcg10(per_user_lt, 'Hybrid')

comparisons = [
    # (protocol, label, A_scores, B_scores)
    ('standard',  'Popularity vs Hybrid',   std_pop,  std_hy),
    ('standard',  'Hybrid vs Content',       std_hy,   std_ct),
    ('standard',  'Hybrid vs CFColdStart',   std_hy,   std_cfcs),
    ('long-tail', 'Hybrid vs Popularity',    lt_hy,    lt_pop),
    ('long-tail', 'Hybrid vs Content',       lt_hy,    lt_ct),
    ('long-tail', 'Hybrid vs CFColdStart',   lt_hy,    lt_cfcs),
]

ht_rows = []
print(f'{"Protocol":<12} {"Comparison":<28} {"delta_mean":>12} {"p_value":>10} {"sig@0.05":>10}')
print('-' * 78)
for protocol, label, a, b in comparisons:
    delta = float((a - b).mean())
    pval  = paired_bootstrap_test(a, b, n_resamples=10_000, seed=42)
    sig   = pval < 0.05
    ht_rows.append({'protocol': protocol, 'comparison': label,
                    'delta_mean': delta, 'p_value': pval,
                    'significant_at_0.05': sig})
    print(f'{protocol:<12} {label:<28} {delta:>12.5f} {pval:>10.4f} {str(sig):>10}')

pd.DataFrame(ht_rows).to_csv(_HT_CSV, index=False)
print(f'\nSaved cold_start_hypothesis_tests.csv')


---
## Section 6 — Visualizations

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

model_order = ['Popularity', 'CFColdStart', 'Content', 'Hybrid']
labels      = ['Popularity', 'CF\nColdStart', 'Content', 'Hybrid']
x = np.arange(len(model_order))
width = 0.35

def ndcg10(df, model):
    r = df[(df['model']==model)&(df['k']==10)&(df['metric']=='ndcg')].iloc[0]
    return r['mean'], r['mean']-r['ci_low'], r['ci_high']-r['mean']

std_vals  = [ndcg10(cs_metrics, m)  for m in model_order]
lt_vals   = [ndcg10(lt_metrics, m)  for m in model_order]

fig, ax = plt.subplots(figsize=(8, 5))
bars_std = ax.bar(x - width/2, [v[0] for v in std_vals],
                  width, yerr=[[v[1] for v in std_vals],[v[2] for v in std_vals]],
                  capsize=4, color='#4C72B0', label='Standard')
bars_lt  = ax.bar(x + width/2, [v[0] for v in lt_vals],
                  width, yerr=[[v[1] for v in lt_vals],[v[2] for v in lt_vals]],
                  capsize=4, color='#DD8452', label='Long-tail (top-100 excluded)')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('NDCG@10')
ax.set_title('Cold-Start: Standard vs Long-Tail Evaluation\n(95% bootstrap CI, n=871 users)')
ax.legend()
ax.set_ylim(bottom=0)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'coldstart_standard_vs_longtail.png', dpi=150)
plt.close(fig)
print('Saved coldstart_standard_vs_longtail.png')


In [ ]:
warm_metrics = pd.read_csv(RESULTS_DIR / 'final_metrics.csv')

# Map warm model names (lowercase) to display names; align with cold-start convention
warm_name_map = {
    'popularity': 'Popularity',
    'cf':         'CFColdStart',  # CF warm == CF; CFColdStart cold is the degenerate version
    'content':    'Content',
    'hybrid':     'Hybrid',
}

def warm_ndcg10(model_key):
    r = warm_metrics[(warm_metrics['model']==model_key)&
                     (warm_metrics['k']==10)&
                     (warm_metrics['metric']=='ndcg')].iloc[0]
    return r['mean'], r['mean']-r['ci_low'], r['ci_high']-r['mean']

protocols = ['Warm (CF/Hybrid)', 'Cold-Standard', 'Cold-Long-tail']
model_keys_warm = ['popularity', 'cf', 'content', 'hybrid']

warm_v  = [warm_ndcg10(k)    for k in model_keys_warm]
std_v   = [ndcg10(cs_metrics, m) for m in model_order]
lt_v    = [ndcg10(lt_metrics, m) for m in model_order]

x      = np.arange(len(model_order))
width  = 0.25
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(9, 5))
for i, (vals, label, color) in enumerate(zip([warm_v, std_v, lt_v], protocols, colors)):
    ax.bar(x + (i-1)*width, [v[0] for v in vals],
           width,
           yerr=[[v[1] for v in vals],[v[2] for v in vals]],
           capsize=4, color=color, label=label)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('NDCG@10')
ax.set_title('NDCG@10 Across All Evaluation Protocols\n(95% bootstrap CI)')
ax.legend()
ax.set_ylim(bottom=0)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'three_protocol_comparison.png', dpi=150)
plt.close(fig)
print('Saved three_protocol_comparison.png')


In [ ]:
# Compute source breakdown across all 871 cold-start users
source_counts = {'both': 0, 'content_only': 0, 'popularity_only': 0}
total_slots   = 0

for uid, d in cs_data.items():
    seed = d['seed_song_id']
    try:
        recs = hybrid.recommend(user_id=None, seed_song=seed, k=10)
        for src in recs['source']:
            source_counts[src] = source_counts.get(src, 0) + 1
            total_slots += 1
    except Exception:
        pass

fracs = {k: v / total_slots for k, v in source_counts.items()}
print('Source breakdown across all users (fraction of top-10 slots):')
for k, v in fracs.items():
    print(f'  {k:<20}: {v:.3f}  ({source_counts[k]:,} slots)')

fig, ax = plt.subplots(figsize=(6, 4))
src_labels = ['Both\n(content+pop)', 'Content\nonly', 'Popularity\nonly']
src_vals   = [fracs.get('both',0), fracs.get('content_only',0), fracs.get('popularity_only',0)]
bar_colors = ['#55A868', '#4C72B0', '#DD8452']
ax.bar(src_labels, src_vals, color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Fraction of top-10 recommendation slots')
ax.set_title('Hybrid Cold-Start: Source Breakdown\n(across all 871 users, k=10)')
ax.set_ylim(0, 1)
for i, v in enumerate(src_vals):
    ax.text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=11)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'hybrid_source_breakdown.png', dpi=150)
plt.close(fig)
print('Saved hybrid_source_breakdown.png')
